# End-to-End CatBoost Workflow — v4 (Allocator Safety Rails)

This notebook is a fully worked example of how to use `portfolio_toolkit` with CatBoost gradient-boosted models.
It trains on **`shared_set_1`** — the full S&P 500 universe (503 tickers) — for maximum diversity and generalisation.

It covers the full workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer a custom notebook-local feature
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. train three CatBoost regressors (return, alpha, volatility)
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow as the final `Felix_Run14` run


## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


## What This Example Is Doing

This notebook is now locked to the final Run 12 feature set selected from the feature ablation review.

The ablation comparison rejected `run11` and `benchmark_only`, and kept the full Run 12 feature set over `pruned` because it had the better risk-adjusted profile: slightly higher Sharpe and Sortino while staying close on return and drawdown.

The goal from here is no more feature experimentation: run this notebook as the final CatBoost candidate with the 43 Run 12 features.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
    log_model_submission,
    validate_feature_frame,
)

print('Imports loaded successfully.')


In [ ]:
# Resolve repo_root robustly — always anchor to this notebook's location.
repo_root = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path().resolve()
# If the above still resolves incorrectly, uncomment and set explicitly:
repo_root = Path(r'C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib')
print('repo_root:', repo_root)
dataset_name = 'shared_set_1'
model_name = 'catboost_sp500'

# Final ablation decision: keep the full Run 12 feature set.
# Run 12 beat Run 11 and benchmark_only, and edged pruned on Sharpe/Sortino.
FINAL_FEATURE_SET = 'run12'
# ── Prediction horizon ───────────────────────────────────────────────────────
# Options: 5 (weekly), 10 (bi-weekly), 21 (monthly)
# Longer horizons = less noise but slower signal; shorter = more trades, higher costs
horizon = 10  # Changed to 10d (bi-weekly) — better signal-to-noise than 21d or 5d

# ── Rebalancing frequency ────────────────────────────────────────────────────
# Options: 1 (daily), 5 (weekly), 21 (monthly)
# Should match or be shorter than horizon for consistency
rebalance_every_n_days = 10  # bi-weekly rebalancing — matches prediction horizon
output_dir = repo_root / 'runs' / 'catboost_sp500_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)

# Concentration safety rails (CRITICAL - fixes Run11 SLS disaster)
# Run11 hidden backtest put 98% into SLS on one date, causing -35% one-day loss.
# ============================================================================
# FINAL MODEL CONFIG - fixed after feature ablation review
# ============================================================================
# Keep the concentration settings from the best Run 12 ablation run.
MAX_WEIGHT_CAP        = 0.02
MIN_ACTIVE_NAMES      = 75
MIN_EFFECTIVE_NAMES   = 25     # minimum 1/sum(w^2)
ENSEMBLE_SEEDS        = 3      # number of seeds for ensemble (real per-row uncertainty)


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

## 2. Add Built-In Toolkit Features

We start with a full built-in feature set.

**Important — leak prevention:** Features are built from the full price history
(all rolling features look backward only, so no future data is used).
However to prevent any subtle leakage from the `dropna()` step interacting with
split boundaries, we split the raw price data first and build features
separately per split, then reassemble. This ensures val/test feature distributions
are never influenced by future price history.


In [ ]:
# ── Split-before-feature-engineering (leak prevention) ──────────────────────
splits_cfg = split_dates(dataset_name, repo_root=repo_root)

train_cutoff = pd.Timestamp("2020-12-31")
val_end      = pd.Timestamp("2021-12-31")
test_start   = pd.Timestamp("2022-01-03")
train_start  = pd.Timestamp("2014-01-02")

# ── Embargo gap: 10 trading days between train end and val start ──────────────
# Prevents target leakage from overlapping forward windows.
# Idea borrowed from Brian's LightGBM model.
EMBARGO_DAYS = 10

# ── Richer base feature set (borrowed from Brian and Hannah) ──────────────────
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_5d',
    'momentum_20d',
    'momentum_60d',
    'momentum_120d',          # NEW: longer-term momentum
    'downside_vol_20d',       # NEW: downside-only volatility
    'upside_vol_20d',         # NEW: upside-only volatility
    'distance_to_20d_high',   # NEW: how far below 20d peak
    'distance_to_60d_high',   # NEW: how far below 60d peak
    'rsi_14',
    'macd_hist',              # NEW: trend acceleration
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'dollar_volume_ratio_20d',# NEW: liquidity proxy
    'excess_return_5d_vs_spy',
    'excess_return_20d_vs_spy',
    'excess_return_60d_vs_spy',
    'relative_momentum_20d_vs_spy',  # NEW: relative strength
    'beta_20d_spy',
    'beta_60d_spy',           # NEW: longer-term beta
    'intraday_range',
]

assert FINAL_FEATURE_SET == 'run12', "This notebook is locked to the final Run 12 feature set."

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
print(f'Final feature set: {FINAL_FEATURE_SET}')
print(f'Base feature count: {len(base_feature_names)}')
display(base_features.head())


## 3. Add New Custom Features In The Notebook

The toolkit supplies the base feature set; everything below is the notebook-local custom portion of the final Run 12 feature set.

We add three categories of custom features:

**Stock-level signals:**
- `mom_vol_ratio` — momentum normalised by volatility
- `trend_spread` — short-term SMA minus long-term SMA
- `quality_signal` — excess return penalised by volatility
- `range_volume_interaction` — intraday range × volume z-score
- `rolling_sharpe_20d` — annualised 20-day rolling Sharpe ratio per ticker

**Volatility regime features:**
- `vol_of_vol` — std of rolling 5d vol over a 20-day window; captures whether vol is stable or erratic
- `drawdown_20d` — current price vs 20-day high; a stock at -25% behaves differently to one at ATH

**Macro / regime features (date-level, same value for all tickers on a given day):**
- `vix_close` — CBOE VIX, the market fear gauge
- `vix_change_5d` — 5-day change in VIX (rising = fear increasing)
- `tlt_momentum_20d` — 20-day return of TLT (Treasury bond ETF, inverse proxy for 10Y yield)
- `rate_regime` — sign of `tlt_momentum_20d`: +1 = rates falling, −1 = rates rising

All features are derived purely from price and volume data — no ticker identity is encoded.
This means the model generalises to any ticker or ETF it has not seen during training,
as long as OHLCV history is available.


In [ ]:
frame = base_features.copy()
eps = 1e-6

# ── Stock-level custom features ──────────────────────────────────────────────
frame['mom_vol_ratio']             = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread']              = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal']            = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction']  = frame['intraday_range'] * frame['volume_zscore_20d']

# ── Volatility regime features ────────────────────────────────────────────────
daily_returns = (
    prices.sort_values(['ticker', 'date'])
    .assign(daily_ret=lambda df: df.groupby('ticker')['close'].pct_change())
)

# rolling_sharpe_20d
rolling_sharpe = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(20).mean() / (s.rolling(20).std(ddof=0) + eps))
)
daily_returns['rolling_sharpe_20d'] = rolling_sharpe * (252 ** 0.5)
sharpe_lookup = daily_returns[['date', 'ticker', 'rolling_sharpe_20d']]
frame = frame.merge(sharpe_lookup, on=['date', 'ticker'], how='left')

# vol_of_vol
vol_of_vol = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(5).std().rolling(20).std())
)
daily_returns['vol_of_vol'] = vol_of_vol
vov_lookup = daily_returns[['date', 'ticker', 'vol_of_vol']]
frame = frame.merge(vov_lookup, on=['date', 'ticker'], how='left')

# drawdown_20d
drawdown_20d = (
    prices.sort_values(['ticker', 'date'])
    .groupby('ticker')['close']
    .transform(lambda s: s / s.rolling(20).max() - 1)
)
drawdown_lookup = (
    prices.sort_values(['ticker', 'date'])[['date', 'ticker']]
    .assign(drawdown_20d=drawdown_20d.values)
)
frame = frame.merge(drawdown_lookup, on=['date', 'ticker'], how='left')

# ── Macro / regime features ───────────────────────────────────────────────────
import yfinance as yf
_start = '2014-01-01'
_end   = '2025-12-31'

def _fetch_close(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    close = df['Close']
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close

vix_raw = _fetch_close("^VIX", _start, _end).rename("vix_close")
tlt_raw = _fetch_close("TLT",  _start, _end).rename("tlt_close")

macro = (
    pd.concat([vix_raw, tlt_raw], axis=1)
    .reset_index()
    .rename(columns={'Date': 'date', 'Datetime': 'date', 'index': 'date'})
    .assign(date=lambda df: pd.to_datetime(df['date']).dt.normalize())
)
macro['vix_change_5d']    = macro['vix_close'].pct_change(5)
macro['tlt_momentum_20d'] = macro['tlt_close'].pct_change(20)
macro['rate_regime']      = macro['tlt_momentum_20d'].apply(lambda x: 1.0 if x >= 0 else -1.0)
macro_feature_names = ['vix_close', 'vix_change_5d', 'tlt_momentum_20d', 'rate_regime']
macro = macro[['date'] + macro_feature_names].dropna()
frame['date'] = pd.to_datetime(frame['date']).dt.normalize()
frame = frame.merge(macro, on='date', how='left')
print('Macro features merged.')

# ── Regime & categorical features ────────────────────────────────────────────
frame['vix_regime'] = pd.cut(
    frame['vix_close'],
    bins=[0, 15, 20, 30, 100],
    labels=[0, 1, 2, 3]
).astype(float)

frame['momentum_regime'] = pd.cut(
    frame['momentum_20d'],
    bins=[-np.inf, -0.05, 0, 0.05, np.inf],
    labels=[0, 1, 2, 3]
).astype(float)

eps2 = 1e-6
daily_ret_std5  = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(5).std())
daily_ret_std20 = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(20).std())
vol_expansion = daily_ret_std5 / (daily_ret_std20 + eps2)
vol_exp_lookup = (
    daily_returns[['date', 'ticker']]
    .assign(vol_expansion_ratio=vol_expansion.values)
)
frame = frame.merge(vol_exp_lookup, on=['date', 'ticker'], how='left')
frame['vol_expanding'] = (frame['vol_expansion_ratio'] > 1.2).astype(float)

print('Regime and categorical features added.')

# ── Cross-sectional rank features (computed AFTER split to avoid leakage) ───
# These rank each stock relative to all others on the SAME date.
# IMPORTANT: ranks are computed separately on train, val, test to avoid
# future dates influencing the rank distribution of past dates.
# We add them to the frame now as NaN placeholders — actual values filled in cell 15.
frame['xs_return_rank']   = np.nan  # rank of return_20d across tickers per date
frame['xs_momentum_rank'] = np.nan  # rank of momentum_20d across tickers per date
frame['xs_sharpe_rank']   = np.nan  # rank of rolling_sharpe_20d across tickers per date
frame['xs_quality_rank']  = np.nan  # rank of quality_signal across tickers per date
print("Cross-sectional rank placeholders added — will be filled per split in cell 15.")

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
    'rolling_sharpe_20d',
    'vol_of_vol',
    'drawdown_20d',
    'vix_close',
    'vix_change_5d',
    'tlt_momentum_20d',
    'rate_regime',
    'xs_return_rank',
    'xs_momentum_rank',
    'xs_sharpe_rank',
    'xs_quality_rank',
    'vix_regime',
    'momentum_regime',
    'vol_expansion_ratio',
    'vol_expanding',
]

all_feature_names = base_feature_names + custom_feature_names
assert len(base_feature_names) == 24
assert len(custom_feature_names) == 19
assert len(all_feature_names) == 43
print(f'Final feature set: {FINAL_FEATURE_SET}')
print(f'Total features: {len(all_feature_names)} ({len(base_feature_names)} base + {len(custom_feature_names)} custom)')
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())


## 4. Build Targets

We fit CatBoost models with the final Run 12 input features for:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This uses the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [ ]:
# ── Build targets for multiple horizons ──────────────────────────────────────
return_targets_5d  = make_forward_return_target(prices, horizon=5)
return_targets_10d = make_forward_return_target(prices, horizon=10)
return_targets_21d = make_forward_return_target(prices, horizon=21)

alpha_targets_5d   = make_forward_alpha_target(prices, horizon=5)
alpha_targets_10d  = make_forward_alpha_target(prices, horizon=10)
alpha_targets_21d  = make_forward_alpha_target(prices, horizon=21)

vol_targets_5d     = make_forward_realized_vol_target(prices, window=5)
vol_targets_10d    = make_forward_realized_vol_target(prices, window=10)
vol_targets_21d    = make_forward_realized_vol_target(prices, window=21)

# Merge all target horizons into the feature frame
target_frame = frame.copy()
for df in [
    return_targets_5d, return_targets_10d, return_targets_21d,
    alpha_targets_5d,  alpha_targets_10d,  alpha_targets_21d,
    vol_targets_5d,    vol_targets_10d,    vol_targets_21d,
]:
    target_frame = target_frame.merge(df, on=['date', 'ticker'], how='left')

# ── Drop NaNs excluding rank columns ─────────────────────────────────────────
# xs_*_rank columns are NaN placeholders at this stage — they get filled
# per split in cell 15 to prevent leakage. Exclude them from dropna().
rank_cols = [c for c in target_frame.columns if c.startswith("xs_")]
cols_to_check = [c for c in target_frame.columns if c not in rank_cols]

target_frame = (
    target_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=cols_to_check)
    .reset_index(drop=True)
)

# ── Active target columns ─────────────────────────────────────────────────────
return_target_col = f"forward_return_{horizon}d"
alpha_target_col  = f"forward_alpha_{horizon}d_vs_spy"
vol_target_col    = f"forward_realized_vol_{horizon}d"

print(f"Active horizon: {horizon}d")
print(f"Return target:  {return_target_col}")
print(f"Alpha target:   {alpha_target_col}")
print(f"Vol target:     {vol_target_col}")
print("Modeling frame shape after dropping nulls:", target_frame.shape)
print("Rank cols still NaN (expected):", target_frame[rank_cols].isna().all().all())
display(target_frame.head())


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. **Splits** the feature frame into train / val / test using the shared date windows
   defined in `datasets.toml` — so every model in the repo is evaluated on the same periods.

2. **Standardizes** features using *only* training-set statistics (mean and std),
   then applies those same statistics to val and test. This prevents data leakage.

3. **Saves the scaler** to disk as a JSON file so it can be reloaded at inference time
   and applied to any ticker — including ETFs not seen during training.
   Because all features are derived from price/volume behavior (not ticker identity),
   the scaler generalises to any OHLCV time series.


In [ ]:
import json as _json

# ── Cross-sectional rank function (defined first so it can be called below) ──
def add_xs_ranks(df):
    """Add cross-sectional percentile ranks per date within this split only.
    Computed per split to prevent leakage across train/val/test boundaries.
    """
    df = df.copy()
    for src_col, rank_col in [
        ('return_20d',         'xs_return_rank'),
        ('momentum_20d',       'xs_momentum_rank'),
        ('rolling_sharpe_20d', 'xs_sharpe_rank'),
        ('quality_signal',     'xs_quality_rank'),
    ]:
        if src_col in df.columns:
            df[rank_col] = df.groupby('date')[src_col].rank(pct=True)
    return df

# ── Override train cutoff to 2020-12-31 ──────────────────────────────────────
# Ensure date column is datetime before filtering
target_frame["date"] = pd.to_datetime(target_frame["date"])

train = target_frame[
    (target_frame["date"] >= pd.Timestamp("2014-01-02")) &
    (target_frame["date"] <= pd.Timestamp("2020-12-31"))
].copy()
# Embargo: skip the first EMBARGO_DAYS trading days after train cutoff
embargo_end = pd.Timestamp("2020-12-31") + pd.Timedelta(days=EMBARGO_DAYS * 2)
val = target_frame[
    (target_frame["date"] >= embargo_end) &
    (target_frame["date"] <= pd.Timestamp("2021-12-31"))
].copy()
test = target_frame[
    (target_frame["date"] >= pd.Timestamp("2022-01-03")) &
    (target_frame["date"] <= pd.Timestamp("2025-12-31"))
].copy()

# Apply ranks per split (no leakage)
train = add_xs_ranks(train)
val   = add_xs_ranks(val)
test  = add_xs_ranks(test)
print("Cross-sectional ranks computed per split.")
print("Overridden splits:")
print("Train rows:", len(train), "| cutoff: 2020-12-31")
print("Val rows:  ", len(val))
print("Test rows: ", len(test))

# ── Rebalancing frequency subsampling ────────────────────────────────────────
def get_rebalance_dates(df, every_n_days):
    """Return every Nth unique date from a frame."""
    dates = sorted(df["date"].unique())
    return dates[::every_n_days]

rebalance_dates = get_rebalance_dates(test, rebalance_every_n_days)
test_rebalance  = test[test["date"].isin(rebalance_dates)].copy()
print(f"\nRebalance every {rebalance_every_n_days} days")
print(f"Full test rows: {len(test)} → Rebalance test rows: {len(test_rebalance)}")
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# ── Build scaler from training data only (no leakage) ────────────────────────
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

output_dir.mkdir(parents=True, exist_ok=True)
scaler_path = output_dir / "feature_scaler.json"
scaler_dict = {
    "feature_names": all_feature_names,
    "means":         train_means.to_dict(),
    "stds":          train_stds.to_dict(),
    "dataset":       dataset_name,
    "feature_set":   FINAL_FEATURE_SET,
    "horizon":       horizon,
    "rebalance_every_n_days": rebalance_every_n_days,
}
with open(scaler_path, "w") as f:
    _json.dump(scaler_dict, f, indent=2)
print(f"Scaler saved to: {scaler_path}")

# ── Standardize splits ───────────────────────────────────────────────────────
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.fillna(0.0).to_numpy(dtype=float)

X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test_rebalance)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test_rebalance[return_target_col].to_numpy(dtype=float)

y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)

print("X_train shape:", X_train.shape)
print("X_test shape (rebalanced):", X_test.shape)
print("Feature count:", X_train.shape[1])
print("NaN values after standardization:", np.isnan(X_train).sum())


In [ ]:
# ── How to reload the scaler for inference on unseen tickers / ETFs ─────────
# This block is for reference — it shows how to apply the saved scaler
# to a completely new dataframe (e.g. ETF test data not in shared_set_2).
#
# import json as _json
# import numpy as np, pandas as pd
# with open("runs/catboost_end_to_end_workflow/feature_scaler.json") as f:
#     scaler = _json.load(f)
#
# saved_means = pd.Series(scaler["means"])
# saved_stds  = pd.Series(scaler["stds"])
# feature_names = scaler["feature_names"]
#
# def standardize_new(df: pd.DataFrame) -> np.ndarray:
#     return ((df[feature_names] - saved_means) / saved_stds).fillna(0.0).to_numpy(dtype=float)
#
# X_new = standardize_new(new_etf_feature_frame)
# preds = return_model.predict(X_new)  # works on any ticker

print('Scaler reload pattern shown above — no action needed in training run.')


## 6. Train CatBoost Models - With Seed Ensemble

We train ENSEMBLE_SEEDS independent CatBoost models per target:

1. Joint return + alpha (MultiRMSE) - 3 seeds, predictions averaged.
2. Vol rank - 3 seeds, predictions averaged.

Per-prediction uncertainty = std across seeds for each row. This replaces the
previous placeholder (sqrt of overall val MSE which was constant for every row).


In [ ]:
from catboost import CatBoostRegressor, Pool

def make_catboost_params(seed):
    return dict(
        iterations=600,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=8.0,
        min_data_in_leaf=50,
        bagging_temperature=1.0,
        rsm=0.8,
        random_seed=seed,
        verbose=False,
    )

SEEDS = list(range(42, 42 + ENSEMBLE_SEEDS))

# Joint return + alpha ensemble
y_train_joint = np.column_stack([y_train_return, y_train_alpha])
y_val_joint   = np.column_stack([y_val_return,   y_val_alpha])

joint_models = []
for seed in SEEDS:
    print(f"Training joint model seed={seed}...")
    m = CatBoostRegressor(**make_catboost_params(seed), loss_function="MultiRMSE")
    m.fit(X_train, y_train_joint, eval_set=Pool(X_val, y_val_joint), early_stopping_rounds=40)
    joint_models.append(m)

joint_val_preds = np.stack([m.predict(X_val) for m in joint_models], axis=0)
joint_val_mean  = joint_val_preds.mean(axis=0)
return_val_mse  = float(((joint_val_mean[:, 0] - y_val_return) ** 2).mean())
alpha_val_mse   = float(((joint_val_mean[:, 1] - y_val_alpha)  ** 2).mean())
print(f"Joint ensemble - return MSE: {return_val_mse:.8f} | alpha MSE: {alpha_val_mse:.8f}")
joint_model = joint_models[0]

# Vol rank ensemble
y_train_vol_rank = pd.Series(y_train_vol).rank(pct=True).to_numpy()
y_val_vol_rank   = pd.Series(y_val_vol).rank(pct=True).to_numpy()

vol_models = []
for seed in SEEDS:
    print(f"Training vol model seed={seed}...")
    m = CatBoostRegressor(**make_catboost_params(seed))
    m.fit(X_train, y_train_vol_rank, eval_set=[(X_val, y_val_vol_rank)], early_stopping_rounds=40)
    vol_models.append(m)

vol_val_preds = np.stack([m.predict(X_val) for m in vol_models], axis=0)
vol_val_mean  = vol_val_preds.mean(axis=0)
vol_val_mse   = float(((vol_val_mean - y_val_vol_rank) ** 2).mean())
print(f"Vol ensemble - val MSE: {vol_val_mse:.8f}")
vol_model = vol_models[0]

fi = pd.DataFrame({
    "feature":    all_feature_names,
    "importance": joint_models[0].get_feature_importance()
}).sort_values("importance", ascending=False)
print("\nTop 10 features by importance:")
display(fi.head(10))
print(f"\nEnsemble trained: {len(joint_models)} joint + {len(vol_models)} vol models.")


## 7. (Skipped — MLP section removed)

The MLP training section from the original template has been removed.
CatBoost handles all three prediction targets above.


In [ ]:
# This cell is intentionally empty.
# The MLP model definition has been replaced by CatBoost in cell 17.
pass


## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [ ]:
# Ensemble predictions - mean across seeds = prediction, std = uncertainty
joint_test_preds  = np.stack([m.predict(X_test) for m in joint_models], axis=0)
joint_test_mean   = joint_test_preds.mean(axis=0)
joint_test_std    = joint_test_preds.std(axis=0)

test_return_pred  = joint_test_mean[:, 0]
test_alpha_pred   = joint_test_mean[:, 1]
test_return_unc   = joint_test_std[:, 0]
test_alpha_unc    = joint_test_std[:, 1]

vol_test_preds    = np.stack([m.predict(X_test) for m in vol_models], axis=0)
test_vol_rank_pred = vol_test_preds.mean(axis=0).clip(0.01, 0.99)

per_row_uncertainty = test_return_unc + 1e-6

predictions = test_rebalance[
    test_rebalance["ticker"] != spec.benchmark_ticker
].loc[:, ["date", "ticker"]].copy().reset_index(drop=True)

predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred[:len(predictions)]
predictions["expected_alpha"]      = test_alpha_pred[:len(predictions)]
predictions["expected_volatility"] = test_vol_rank_pred[:len(predictions)]
predictions["uncertainty"]         = per_row_uncertainty[:len(predictions)]

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

print(f"Prediction frame shape: {predictions.shape}")
print(f"Uncertainty stats - min: {predictions['uncertainty'].min():.6f} | "
      f"median: {predictions['uncertainty'].median():.6f} | "
      f"max: {predictions['uncertainty'].max():.6f}")
print(f"Rebalancing every {rebalance_every_n_days} days | {len(predictions['date'].unique())} dates")
display(predictions.head())


In [ ]:
# ── Empirical test: does weights_from_predictions_risk_adjusted use uncertainty? ──
# We cannot inspect the toolkit source directly. Instead we run the builder twice:
#   - Once with the real per-row uncertainty
#   - Once with uncertainty set to a constant
# If the resulting weights differ, the builder uses uncertainty for sizing.
# If they are identical, uncertainty is ignored and we should switch builders
# or use a custom sizing scheme to actually leverage the uncertainty signal.

predictions_real_unc = predictions.copy()

predictions_flat_unc = predictions.copy()
predictions_flat_unc["uncertainty"] = predictions["uncertainty"].mean()

_pf_real = weights_from_predictions_risk_adjusted(
    predictions_real_unc,
    dataset_name=dataset_name,
    strategy_name="diag_real_unc",
)
_pf_flat = weights_from_predictions_risk_adjusted(
    predictions_flat_unc,
    dataset_name=dataset_name,
    strategy_name="diag_flat_unc",
)

# Align columns and dates, then compute max absolute difference
_common_cols = sorted(set(_pf_real.weights.columns) & set(_pf_flat.weights.columns))
_common_idx  = _pf_real.weights.index.intersection(_pf_flat.weights.index)
_diff = (
    _pf_real.weights.loc[_common_idx, _common_cols].fillna(0)
    - _pf_flat.weights.loc[_common_idx, _common_cols].fillna(0)
)
_max_abs_diff = float(_diff.abs().max().max())

print(f"Max absolute weight difference between real vs flat uncertainty: {_max_abs_diff:.8f}")
if _max_abs_diff < 1e-8:
    print("\nRESULT: weights_from_predictions_risk_adjusted does NOT use uncertainty.")
    print("        Real uncertainty is being computed and logged but does not affect sizing.")
    print("        Consider switching to a builder that uses it, or implementing custom sizing.")
    UNCERTAINTY_USED_BY_BUILDER = False
else:
    print("\nRESULT: weights_from_predictions_risk_adjusted DOES use uncertainty.")
    print("        Real uncertainty is flowing into sizing as intended.")
    UNCERTAINTY_USED_BY_BUILDER = True


## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In [ ]:
# Build portfolio with toolkit risk-adjusted weight builder
# All concentration safety rails are enforced downstream in the patched validator
# (cell 26) which caps weights and falls back to equal weight if too few names.
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print("Portfolio built with toolkit risk-adjusted weights.")
print(f"Weights frame shape: {validated_weights.shape}")
display(validated_weights.head())


## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [ ]:
# Validator patch with concentration safety rails - fixes Run11 SLS disaster
# Run11 had 1 date with 98% in SLS, 2 dates >50% single name, 6 dates >10%.
# This patch enforces:
#   1. Inf/NaN handled gracefully
#   2. Empty/zero rows fall back to equal weight
#   3. Per-stock cap via iterative redistribution
#   4. Fallback to equal weight if too few names survive masking
#   5. Final row sums always exactly 1.0
import portfolio_toolkit.validation as _val
import portfolio_toolkit.backtest   as _bt

_original_validate = _val.validate_weights_frame

def _apply_cap_with_redistribution(row, cap):
    """Cap weights at `cap`, redistribute excess proportionally to under-cap names."""
    w = row.astype(float).copy()
    if w.sum() <= 0:
        return w
    w = w / w.sum()
    n_active = (w > 0).sum()
    effective_cap = max(cap, 1.0 / max(n_active, 1))
    for _ in range(100):
        over = w > effective_cap
        if not over.any():
            break
        excess = float((w.loc[over] - effective_cap).sum())
        w.loc[over] = effective_cap
        under = (~over) & (w > 0)
        under_total = float(w.loc[under].sum())
        if under_total <= 0 or excess <= 1e-12:
            break
        w.loc[under] = w.loc[under] + excess * (w.loc[under] / under_total)
    return w / w.sum() if w.sum() > 0 else w

def _patched_validate(df, dataset_name=None, repo_root=None):
    n_cols = df.shape[1]
    df = df.replace([float("inf"), float("-inf")], 0.0)

    row_sums = df.sum(axis=1)
    zero_rows = row_sums.abs() < 1e-8
    if zero_rows.any():
        df.loc[zero_rows] = 1.0 / n_cols

    df = df.div(df.sum(axis=1), axis=0).fillna(1.0 / n_cols)

    capped = df.apply(lambda r: _apply_cap_with_redistribution(r, MAX_WEIGHT_CAP), axis=1)

    active_counts = (capped > 0).sum(axis=1)
    too_few = active_counts < MIN_ACTIVE_NAMES
    if too_few.any():
        capped.loc[too_few] = 1.0 / n_cols

    capped = capped.div(capped.sum(axis=1), axis=0)
    return capped

_val.validate_weights_frame = _patched_validate
_bt.validate_weights_frame  = _patched_validate

print("validate_weights_frame patched with safety rails:")
print(f"  - max weight cap:   {MAX_WEIGHT_CAP * 100:.1f}%")
print(f"  - min active names: {MIN_ACTIVE_NAMES}")
print(f"  - equal-weight fallback when too few names survive masking")


In [ ]:
# ── Clean prices and filter tickers with missing data ────────────────────────
from portfolio_toolkit.data import load_prices as _lp
from portfolio_toolkit.backtest import _pivot_prices as _pp
import portfolio_toolkit.data as _data_module
import portfolio_toolkit.backtest as _bt_module

_prices_full = _lp(dataset_name, repo_root=repo_root)
_price_wide  = _pp(_prices_full)
_price_wide_filled = _price_wide.ffill(limit=10)

# Find tickers with persistent NaN prices after forward-fill
_bad_tickers = set(
    _price_wide_filled.columns[_price_wide_filled.isna().any()].tolist()
)
_known_bad = {'FISV', 'BF-B', 'BRK-B'}
_bad_tickers = _bad_tickers | _known_bad

if _bad_tickers:
    print(f"Excluding {len(_bad_tickers)} tickers with NaN prices: {sorted(_bad_tickers)}")

# ── Memory optimisation: limit universe to clean tickers only ─────────────────
# shared_set_1 has 503 tickers. The equal-weight benchmark allocates a
# (n_days × n_tickers) matrix which causes MemoryError on machines with
# limited RAM. We cap the universe at the clean tickers only, which also
# removes the NaN problem in one step.
_clean_tickers = [
    t for t in spec.tickers
    if t not in _bad_tickers and t in _price_wide_filled.columns
]
print(f"Clean universe: {len(_clean_tickers)} tickers (from {len(spec.tickers)} in spec)")

# Patch load_prices to return only clean tickers
_prices_clean = _prices_full[
    _prices_full["ticker"].isin(_clean_tickers)
].copy()
_original_load_prices = _data_module.load_prices
def _patched_load_prices(dataset_name, repo_root=None):
    return _prices_clean
_data_module.load_prices = _patched_load_prices
_bt_module.load_prices   = _patched_load_prices

# Also patch baseline_weights to use clean tickers
import portfolio_toolkit.baselines as _baselines_module
import portfolio_toolkit.backtest as _bt_module2
_original_baseline = _baselines_module.baseline_weights
def _patched_baseline(dataset_name_or_spec, strategy_name, split=None, repo_root=None):
    # Pass split only if it is not None — slice_split raises KeyError on None
    kwargs = {"repo_root": repo_root}
    if split is not None:
        kwargs["split"] = split
    result = _original_baseline(dataset_name_or_spec, strategy_name, **kwargs)
    # Keep only clean tickers in the baseline weights
    clean_cols = [c for c in result.weights.columns if c in _clean_tickers]
    w = result.weights[clean_cols].copy()
    row_sums = w.sum(axis=1).replace(0, 1.0)
    result.weights = w.div(row_sums, axis=0)
    return result
_baselines_module.baseline_weights = _patched_baseline
_bt_module2.baseline_weights       = _patched_baseline

# ── Renormalize portfolio weights over clean tickers only ─────────────────────
raw_w    = portfolio.weights[
    [t for t in portfolio.weights.columns if t in _clean_tickers]
].copy()
row_sums = raw_w.sum(axis=1).replace(0, 1.0)
renorm_w = raw_w.div(row_sums, axis=0)

from portfolio_toolkit.portfolio import PortfolioWeights
portfolio_renorm = PortfolioWeights(
    weights=renorm_w,
    strategy_name=portfolio.strategy_name,
    dataset_name=dataset_name,
)

result         = backtest_weights(dataset_name, portfolio_renorm, repo_root=repo_root)
# Downstream diagnostics and MLflow logging should use the final aligned weights
# returned by the backtest after concentration safety rails are applied.
validated_weights = result.weights.copy()
metrics        = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

# Restore originals
_data_module.load_prices       = _original_load_prices
_bt_module.load_prices         = _original_load_prices
_baselines_module.baseline_weights = _original_baseline
_bt_module2.baseline_weights       = _original_baseline

metrics_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in sorted(metrics.items())]
).sort_values("metric").reset_index(drop=True)

display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])


## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUBMISSION BLOCK — Required functions, artifact saving, feature/scaler refs
# ══════════════════════════════════════════════════════════════════════════════

# ── Required Function 1: build_model_features ────────────────────────────────
def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild the exact feature table used by the model from raw OHLCV price data.
    Accepts: date, ticker, open, high, low, close, adj_close, volume.
    Returns: date, ticker, + all feature columns in all_feature_names order.
    Never uses forward-looking columns (forward_return_*, forward_alpha_*, etc).
    """
    eps = 1e-6

    # ── Base toolkit features ─────────────────────────────────────────────────
    frame = build_features(prices, feature_names=base_feature_names)

    # ── Stock-level custom features ───────────────────────────────────────────
    frame["mom_vol_ratio"]            = frame["momentum_20d"] / (frame["vol_20d"].abs() + eps)
    frame["trend_spread"]             = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["quality_signal"]           = frame["excess_return_20d_vs_spy"] - 0.5 * frame["vol_20d"]
    frame["range_volume_interaction"] = frame["intraday_range"] * frame["volume_zscore_20d"]

    # ── Volatility regime features ────────────────────────────────────────────
    daily_returns = (
        prices.sort_values(["ticker", "date"])
        .assign(daily_ret=lambda df: df.groupby("ticker")["close"].pct_change())
    )
    rolling_sharpe = (
        daily_returns.groupby("ticker")["daily_ret"]
        .transform(lambda s: s.rolling(20).mean() / (s.rolling(20).std(ddof=0) + eps))
    )
    daily_returns["rolling_sharpe_20d"] = rolling_sharpe * (252 ** 0.5)
    frame = frame.merge(
        daily_returns[["date", "ticker", "rolling_sharpe_20d"]], on=["date", "ticker"], how="left"
    )
    vol_of_vol = (
        daily_returns.groupby("ticker")["daily_ret"]
        .transform(lambda s: s.rolling(5).std().rolling(20).std())
    )
    daily_returns["vol_of_vol"] = vol_of_vol
    frame = frame.merge(
        daily_returns[["date", "ticker", "vol_of_vol"]], on=["date", "ticker"], how="left"
    )
    drawdown_20d = (
        prices.sort_values(["ticker", "date"])
        .groupby("ticker")["close"]
        .transform(lambda s: s / s.rolling(20).max() - 1)
    )
    drawdown_lookup = (
        prices.sort_values(["ticker", "date"])[["date", "ticker"]]
        .assign(drawdown_20d=drawdown_20d.values)
    )
    frame = frame.merge(drawdown_lookup, on=["date", "ticker"], how="left")

    # ── Macro / regime features ───────────────────────────────────────────────
    # SPY used only as feature context — never as a tradable ticker.
    _start = prices["date"].min().strftime("%Y-%m-%d")
    _end   = prices["date"].max().strftime("%Y-%m-%d")

    def _fetch_close(ticker, start, end):
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        close = df["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        return close

    vix_raw = _fetch_close("^VIX", _start, _end).rename("vix_close")
    tlt_raw = _fetch_close("TLT",  _start, _end).rename("tlt_close")
    macro = (
        pd.concat([vix_raw, tlt_raw], axis=1).reset_index()
        .rename(columns={"Date": "date", "Datetime": "date", "index": "date"})
        .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
    )
    macro["vix_change_5d"]    = macro["vix_close"].pct_change(5)
    macro["tlt_momentum_20d"] = macro["tlt_close"].pct_change(20)
    macro["rate_regime"]      = macro["tlt_momentum_20d"].apply(lambda x: 1.0 if x >= 0 else -1.0)
    macro = macro[["date", "vix_close", "vix_change_5d", "tlt_momentum_20d", "rate_regime"]].dropna()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame = frame.merge(macro, on="date", how="left")

    # ── Regime categorical features ───────────────────────────────────────────
    frame["vix_regime"] = pd.cut(
        frame["vix_close"], bins=[0, 15, 20, 30, 100], labels=[0, 1, 2, 3]
    ).astype(float)
    frame["momentum_regime"] = pd.cut(
        frame["momentum_20d"], bins=[-np.inf, -0.05, 0, 0.05, np.inf], labels=[0, 1, 2, 3]
    ).astype(float)
    daily_ret_std5  = daily_returns.groupby("ticker")["daily_ret"].transform(lambda s: s.rolling(5).std())
    daily_ret_std20 = daily_returns.groupby("ticker")["daily_ret"].transform(lambda s: s.rolling(20).std())
    vol_expansion = daily_ret_std5 / (daily_ret_std20 + eps)
    vol_exp_lookup = daily_returns[["date", "ticker"]].assign(vol_expansion_ratio=vol_expansion.values)
    frame = frame.merge(vol_exp_lookup, on=["date", "ticker"], how="left")
    frame["vol_expanding"] = (frame["vol_expansion_ratio"] > 1.2).astype(float)

    # ── Cross-sectional rank features ─────────────────────────────────────────
    for src_col, rank_col in [
        ("return_20d",         "xs_return_rank"),
        ("momentum_20d",       "xs_momentum_rank"),
        ("rolling_sharpe_20d", "xs_sharpe_rank"),
        ("quality_signal",     "xs_quality_rank"),
    ]:
        if src_col in frame.columns:
            frame[rank_col] = frame.groupby("date")[src_col].rank(pct=True)

    return validate_feature_frame(frame)


# ── Required Function 2: predict_from_prices ─────────────────────────────────
# Exact spec signature: predict_from_prices(model, prices, dates=None, tickers=None)
# model = dict with keys "joint_model" and "vol_model", or bare joint_model.
# Scaler is reloaded from the saved feature_scaler.json artifact.
_saved_scaler_path = scaler_path  # captured at definition time

def predict_from_prices(
    model,
    prices: pd.DataFrame,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    """
    Rebuild features, apply training scaler, run inference, return predictions.
    Fully self-contained — reloads scaler from disk, no live memory needed.
    """
    import json as _j
    with open(_saved_scaler_path) as f:
        _scaler = _j.load(f)
    _means        = pd.Series(_scaler["means"])
    _stds         = pd.Series(_scaler["stds"])
    _feature_names = _scaler["feature_names"]
    _horizon       = _scaler["horizon"]

    # Accept dict {"joint_model": ..., "vol_model": ...} or bare joint_model
    if isinstance(model, dict):
        _models = model
    else:
        from catboost import CatBoostRegressor as _CB
        _vm = _CB()
        _vm.load_model(str(_saved_scaler_path).replace("feature_scaler.json", "vol_model.cbm"))
        _models = {"joint_model": model, "vol_model": _vm}

    features = build_model_features(prices)

    if dates is not None:
        dates = pd.DatetimeIndex(pd.to_datetime(dates))
        features = features.loc[features["date"].isin(dates)].copy()

    if tickers is not None:
        tickers = [str(t).upper() for t in tickers]
        features = features.loc[features["ticker"].isin(tickers)].copy()

    # Drop rows with NaN in any non-rank feature (rank cols filled via groupby)
    non_rank_features = [f for f in _feature_names if not f.startswith("xs_")]
    scoring_frame = features.dropna(subset=non_rank_features).reset_index(drop=True)

    # Apply training scaler
    X = ((scoring_frame[_feature_names] - _means) / _stds).fillna(0.0).to_numpy(dtype=float)

    # Inference
    joint_pred  = _models["joint_model"].predict(X)
    vol_pred    = _models["vol_model"].predict(X).clip(0.01, 0.99)

    predictions = scoring_frame[["date", "ticker"]].copy()
    predictions["horizon"]             = _horizon
    predictions["expected_return"]     = joint_pred[:, 0]
    predictions["expected_alpha"]      = joint_pred[:, 1]
    predictions["expected_volatility"] = vol_pred

    return validate_prediction_frame(predictions, horizon=_horizon)


# ── Save model artifacts (.cbm format for CatBoost) ──────────────────────────
output_dir.mkdir(parents=True, exist_ok=True)
joint_model_path = output_dir / "joint_model.cbm"
vol_model_path   = output_dir / "vol_model.cbm"
joint_model.save_model(str(joint_model_path), format="cbm")
vol_model.save_model(str(vol_model_path),   format="cbm")

print("✓ build_model_features defined")
print("✓ predict_from_prices defined")
print(f"✓ joint_model saved: {joint_model_path}")
print(f"✓ vol_model saved:   {vol_model_path}")
print(f"✓ scaler saved:      {scaler_path}")
print(f"✓ feature_names: {len(all_feature_names)} features | horizon: {horizon}d | rebalance: every_{rebalance_every_n_days}_trading_days")


In [ ]:
mlflow_layout = init_mlflow(repo_root)
print("MLflow tracking URI:", mlflow_layout["tracking_uri"])

notebook_path = Path(__file__).resolve() if "__file__" in dir() else Path("catboost_full_model.ipynb").resolve()
rebalance_frequency = f"every_{rebalance_every_n_days}_trading_days"

with start_run(
    run_name="Felix_Run14",
    dataset_name=dataset_name,
    tags={
        "workflow":           "catboost_sp500_workflow",
        "model_family":       "catboost",
        "prediction_horizon": str(horizon),
        "feature_set":        FINAL_FEATURE_SET,
    },
    repo_root=repo_root,
):
    import mlflow

    _w = validated_weights
    _max_w  = float(_w.max(axis=1).max())
    _min_n  = int((_w > 0).sum(axis=1).min())
    _min_eff = float((1.0 / (_w ** 2).sum(axis=1)).min())
    _top10_max = float(_w.apply(lambda r: r.nlargest(10).sum(), axis=1).max())

    mlflow.log_metrics({
        "concentration_max_single_weight": _max_w,
        "concentration_min_active_names":  _min_n,
        "concentration_min_effective_n":   _min_eff,
        "concentration_max_top10":         _top10_max,
    })

    mlflow.log_params({
        "model_name":             model_name,
        "dataset_name":           dataset_name,
        "horizon":                horizon,
        "rebalance_frequency":    rebalance_frequency,
        "feature_set":            FINAL_FEATURE_SET,
        "feature_count":          len(all_feature_names),
        "base_feature_count":     len(base_feature_names),
        "custom_feature_count":   len(custom_feature_names),
        "feature_list":           ",".join(all_feature_names),
        "train_cutoff":           "2020-12-31",
        "catboost_iterations":    500,
        "catboost_lr":            0.03,
        "catboost_depth":         6,
        "catboost_l2":            5.0,
        "catboost_min_data_leaf": 20,
        "loss_function":          "MultiRMSE",
        "vol_target":             "cross_sectional_rank",
        "portfolio_builder":      "weights_from_predictions_risk_adjusted",
        "cost_bps":               spec.cost_bps,
        "max_weight_cap":         MAX_WEIGHT_CAP,
        "min_active_names":       MIN_ACTIVE_NAMES,
        "min_effective_names":    MIN_EFFECTIVE_NAMES,
        "ensemble_seeds":         ENSEMBLE_SEEDS,
        "uncertainty_used_by_builder": UNCERTAINTY_USED_BY_BUILDER,
    })

    # Log scaler as artifact so preprocessing is fully reconstructable
    mlflow.log_artifact(str(scaler_path), artifact_path="model_submission/artifacts")

    # ── log_model_submission — spec-compliant non-Torch CatBoost bundle ──────
    log_model_submission(
        {
            "joint_model": str(joint_model_path),
            "vol_model":   str(vol_model_path),
        },
        model_name=model_name,
        model_family="catboost",
        feature_names=all_feature_names,
        target=return_target_col,
        horizon=horizon,
        rebalance_frequency=rebalance_frequency,
        preprocessing={
            "scaler":           "train_mean_std",
            "means":            train_means.to_dict(),
            "stds":             train_stds.to_dict(),
            "fill_policy":      "fillna_zero_after_scaling",
            "scaler_artifact":  "feature_scaler.json",
        },
        model_config={
            "artifact_format":    "cbm",
            "architecture":       "CatBoostRegressor",
            "loss_function":      "MultiRMSE",
            "joint_targets":      ["expected_return", "expected_alpha"],
            "vol_model":          "CatBoostRegressor_vol_rank",
            "input_dim":          len(all_feature_names),
            "iterations":         500,
            "learning_rate":      0.03,
            "depth":              6,
            "l2_leaf_reg":        5.0,
            "min_data_in_leaf":   20,
            "portfolio_builder":  "weights_from_predictions_risk_adjusted",
            "required_functions": ["build_model_features", "predict_from_prices"],
        },
        source_files=[str(notebook_path)],
        notes=(
            "CatBoost model trained on shared_set_1 (S&P 500). "
            "Joint MultiRMSE model predicts return + alpha simultaneously. "
            "Separate vol rank model predicts cross-sectional volatility percentile. "
            "Final Run 12 feature set selected after feature ablation review. "
            "Features include base toolkit features, custom stock-level signals, "
            "macro regime features (VIX/TLT), regime categoricals, and cross-sectional ranks. "
            "Rebalances every 10 trading days. "
            "Scaler reconstructable from feature_scaler.json artifact."
        ),
    )

    log_predictions(predictions)
    log_portfolio(portfolio_renorm)
    log_backtest(result)

print(f"MLflow run Felix_Run14 logged successfully.")


## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:
print("Top-level metrics:")
for key, value in sorted(result.metrics.items()):
    print(f"  {key}: {value:.6f}")

# Concentration diagnostics - compare against Run11 (which had 98% in one name)
w = validated_weights
max_per_row    = w.max(axis=1)
active_per_row = (w > 0).sum(axis=1)
eff_n_per_row  = 1.0 / (w ** 2).sum(axis=1)
top10_per_row  = w.apply(lambda r: r.nlargest(10).sum(), axis=1)

print("\nConcentration diagnostics:")
print(f"  max single-name weight - max: {max_per_row.max():.4f} | mean: {max_per_row.mean():.4f}")
print(f"  active names (>0)      - min: {int(active_per_row.min())} | mean: {active_per_row.mean():.1f}")
print(f"  effective N (1/sum w^2)- min: {eff_n_per_row.min():.1f} | mean: {eff_n_per_row.mean():.1f}")
print(f"  top-10 concentration   - max: {top10_per_row.max():.4f} | mean: {top10_per_row.mean():.4f}")
print(f"  dates breaching {MAX_WEIGHT_CAP*100:.1f}% cap:  {int((max_per_row > MAX_WEIGHT_CAP + 1e-6).sum())} of {len(w)}")
print(f"  dates below {MIN_ACTIVE_NAMES} active names: {int((active_per_row < MIN_ACTIVE_NAMES).sum())} of {len(w)}")


In [ ]:
# Final submission validation + CONCENTRATION ASSERTIONS

assert joint_model_path.exists(), f"FAIL: joint_model.cbm missing at {joint_model_path}"
assert vol_model_path.exists(),   f"FAIL: vol_model.cbm missing at {vol_model_path}"
assert scaler_path.exists(),      f"FAIL: feature_scaler.json missing at {scaler_path}"

assert callable(build_model_features), "FAIL: build_model_features not defined"

import inspect
_sig = inspect.signature(predict_from_prices)
assert list(_sig.parameters.keys()) == ["model", "prices", "dates", "tickers"], \
    f"FAIL: predict_from_prices signature mismatch: {list(_sig.parameters.keys())}"

_model_feature_count = len(joint_model.feature_names_)
assert len(all_feature_names) == _model_feature_count, \
    f"FAIL: feature count mismatch - {len(all_feature_names)} defined vs {_model_feature_count} in model"

import json as _j
with open(scaler_path) as _f:
    _sc = _j.load(_f)
assert _sc["feature_names"] == all_feature_names, "FAIL: scaler feature_names mismatch"
assert _sc.get("feature_set") == FINAL_FEATURE_SET, "FAIL: scaler feature_set mismatch"

assert isinstance(horizon, int) and horizon > 0
assert rebalance_frequency == f"every_{rebalance_every_n_days}_trading_days"

assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert Path(artifact_paths["quantstats_report"]).exists()
assert validated_weights.index.is_monotonic_increasing

# CONCENTRATION ASSERTIONS - fail loudly if safety rails were breached
w = validated_weights
max_per_row    = w.max(axis=1)
active_per_row = (w > 0).sum(axis=1)
eff_n_per_row  = 1.0 / (w ** 2).sum(axis=1)

TOLERANCE = 1e-4
assert (max_per_row <= MAX_WEIGHT_CAP + TOLERANCE).all(), \
    f"FAIL: max single-name weight {max_per_row.max():.4f} exceeds cap {MAX_WEIGHT_CAP}"

assert (active_per_row >= MIN_ACTIVE_NAMES).all(), \
    f"FAIL: dates with fewer than {MIN_ACTIVE_NAMES} active names: {int((active_per_row < MIN_ACTIVE_NAMES).sum())}"

assert (eff_n_per_row >= MIN_EFFECTIVE_NAMES).all(), \
    f"FAIL: dates with effective N below {MIN_EFFECTIVE_NAMES}: {int((eff_n_per_row < MIN_EFFECTIVE_NAMES).sum())}"

print("OK Model artifacts on disk")
print("OK build_model_features defined")
print("OK predict_from_prices signature correct")
print(f"OK feature set: {FINAL_FEATURE_SET}")
print(f"OK feature_names: {len(all_feature_names)} features match model")
print("OK Preprocessing reconstructable")
print(f"OK Horizon: {horizon} | Rebalance: {rebalance_frequency}")
print("OK Backtest metrics + report + weights all valid")
print()
print("--- CONCENTRATION SAFETY RAILS HELD ---")
print(f"OK max single-name weight never exceeded {MAX_WEIGHT_CAP*100:.1f}% (actual: {max_per_row.max():.4f})")
print(f"OK active names never below {MIN_ACTIVE_NAMES} (actual min: {int(active_per_row.min())})")
print(f"OK effective N never below {MIN_EFFECTIVE_NAMES} (actual min: {eff_n_per_row.min():.1f})")
print()
print("=" * 60)
print(f"SUBMISSION READY - Felix_Run14")
print("=" * 60)
